# 🎮 Minecraft AI Builder - Text-to-Build

Generate Minecraft builds from text prompts like **"medieval castle with towers"**

## ✨ What This Does:

1. Trains AI models (~35-48 hours total)
2. Generates text descriptions for dataset using Gemini
3. Learns text-to-build mapping
4. Generates builds from your prompts

## 🚀 How to Use:

1. **Set your Gemini API key** in the Configuration cell below
2. **Click Runtime → Run all**
3. **Wait ~35-48 hours** (or pause and resume)
4. **Download generated builds** at the end

That's it! Everything runs automatically.

---
# ⚙️ Configuration

**REQUIRED:** Get your free Gemini API key at: https://makersuite.google.com/app/apikey

In [ ]:
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"

if GEMINI_API_KEY == "YOUR_GEMINI_API_KEY_HERE":
    raise ValueError("⚠️  Please set your Gemini API key above! Get it at: https://makersuite.google.com/app/apikey")

print(f"✓ API Key configured: {GEMINI_API_KEY[:20]}...")

---
# 🚀 Automatic Setup

In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'

!git clone https://github.com/GogaGogich123/Ai.git
%cd Ai
!git checkout capy/cap-1-bc3cdacc

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
print("✓ Dependencies installed")
print("✓ Package installed")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("✓ GPU ready for training")
else:
    print("⚠️  WARNING: No GPU detected!")
    print("Go to: Runtime → Change runtime type → Hardware accelerator → GPU")
    raise RuntimeError("GPU required for training")

In [ ]:
!python test_training.py

---
# 🎯 Stage 1: VQ-VAE + AI Descriptions

**Time:** ~9-14 hours

This stage:
- Downloads builds from BuildPaste
- Generates AI descriptions with Gemini
- Trains VQ-VAE compression model

**Progress will be shown below. You can close the tab and come back later.**

In [ ]:
!python mcbuilder/train_improved_vqvae.py \
    --cache_dir ./data/cache \
    --checkpoint_dir ./checkpoints_improved \
    --chunk_size 32 \
    --overlap 4 \
    --min_blocks 800 \
    --max_blocks 50000 \
    --batch_size 4 \
    --num_workers 2 \
    --embedding_dim 128 \
    --num_embeddings 1024 \
    --num_res_blocks 3 \
    --lr 1e-4 \
    --epochs 100 \
    --save_every 10 \
    --generate_descriptions \
    --gemini_api_key $GEMINI_API_KEY \
    --description_language en

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/minecraft_ai_checkpoints
!cp -r ./checkpoints_improved /content/drive/MyDrive/minecraft_ai_checkpoints/
!cp -r ./data/cache /content/drive/MyDrive/minecraft_ai_checkpoints/

print("✓ Stage 1 complete! Checkpoint saved to Google Drive.")

---
# 🎯 Stage 2: Text-Conditioned Diffusion

**Time:** ~15-20 hours

This stage:
- Loads pretrained CLIP text encoder
- Trains diffusion with cross-attention
- Learns to generate from text prompts

In [ ]:
!python mcbuilder/train_text_to_build.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --cache_dir ./data/cache \
    --checkpoint_dir ./checkpoints_text_to_build \
    --text_encoder_type clip \
    --context_dim 512 \
    --model_channels 128 \
    --num_res_blocks 2 \
    --attention_resolutions 4 8 \
    --channel_mult 1 2 4 8 \
    --num_heads 8 \
    --dropout 0.1 \
    --timesteps 1000 \
    --chunk_size 32 \
    --batch_size 4 \
    --num_workers 2 \
    --epochs 100 \
    --learning_rate 1e-4 \
    --save_every 10

In [ ]:
!cp -r ./checkpoints_text_to_build /content/drive/MyDrive/minecraft_ai_checkpoints/

print("✓ Stage 2 complete! Text-to-build model ready.")

---
# ✨ Generate Builds from Text!

Training complete! Now generate builds from prompts.

In [ ]:
!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "medieval stone castle with tall towers and fortified walls" \
    --size 64,48,64 \
    --guidance_scale 8.0 \
    --num_samples 3 \
    --validate \
    --output medieval_castle.litematic

In [ ]:
!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "cozy cottage with oak planks stone fireplace and wooden furniture" \
    --size 24,20,24 \
    --guidance_scale 7.5 \
    --validate \
    --output cozy_cottage.litematic

In [ ]:
!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "modern suburban house with white concrete walls and large glass windows" \
    --size 32,24,32 \
    --guidance_scale 7.5 \
    --validate \
    --output modern_house.litematic

In [ ]:
!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "fantasy treehouse with wooden platforms bridges and leaf decorations" \
    --size 32,40,32 \
    --guidance_scale 8.0 \
    --validate \
    --output treehouse.litematic

In [ ]:
!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "traditional japanese pagoda with wooden beams and curved roofs" \
    --size 32,48,32 \
    --guidance_scale 8.5 \
    --validate \
    --output pagoda.litematic

---
## 📥 Download Generated Builds

In [ ]:
from google.colab import files
import os

print("Downloading all .litematic files...\n")

for filename in os.listdir('.'):
    if filename.endswith('.litematic'):
        print(f"📦 {filename}")
        files.download(filename)

print("\n✓ All builds downloaded!")
print("\nTo use in Minecraft:")
print("1. Install Litematica mod")
print("2. Place .litematic files in .minecraft/schematics/")
print("3. Load in-game with M key")

---
## 🎨 Generate Your Own Build (After Training)

Edit the prompt below and run to generate custom builds!

In [ ]:
YOUR_PROMPT = "medieval fortress with stone walls"
SIZE = "32,32,32"
GUIDANCE = 7.5

print(f"🎮 Generating: {YOUR_PROMPT}")
print(f"Size: {SIZE}")
print(f"Guidance: {GUIDANCE}\n")

!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "{YOUR_PROMPT}" \
    --size {SIZE} \
    --guidance_scale {GUIDANCE} \
    --num_samples 3 \
    --validate \
    --output custom_build.litematic

files.download('custom_build.litematic')

---
## 💡 Prompt Examples

**Architecture:**
- "medieval stone castle with tall towers and fortified walls"
- "modern house with glass windows and concrete structure"
- "japanese pagoda with wooden beams and curved roofs"
- "gothic cathedral with stained glass and stone arches"

**Fantasy:**
- "fantasy treehouse with wooden platforms and bridges"
- "wizard tower with magical elements and bookshelves"
- "elven palace with white marble and nature integration"

**Functional:**
- "blacksmith workshop with anvils and furnaces"
- "cozy library with bookshelves and reading area"
- "medieval tavern with wooden interior and bar"

**Tips:**
- Be specific about style and materials
- Mention key features (towers, bridges, windows)
- Use Minecraft terms (planks, cobblestone, glass)
- Keep it 5-15 words

## 📚 Full Documentation

- [TEXT_TO_BUILD.md](https://github.com/GogaGogich123/Ai/blob/capy/cap-1-bc3cdacc/TEXT_TO_BUILD.md) - Complete guide
- [EXAMPLES.md](https://github.com/GogaGogich123/Ai/blob/capy/cap-1-bc3cdacc/EXAMPLES.md) - More examples
- [README.md](https://github.com/GogaGogich123/Ai/blob/capy/cap-1-bc3cdacc/README.md) - Project overview